# Groundwater Potential Mapping for Kano State, Nigeria

## GIS-Based Multi-Criteria Decision Analysis (MCDA)

This notebook develops a groundwater potential map for Kano State, Nigeria using GIS-based multi-criteria analysis. Environmental, topographic, hydrological, soil and land-cover factors are prepared as thematic layers, reclassified to a common suitability scale, combined using a weighted overlay, and classified into five groundwater-potential zones.

### Objectives
1. Identify environmental and geological factors influencing groundwater occurrence.
2. Generate thematic layers representing groundwater-controlling factors.
3. Apply GIS-based Multi-Criteria Decision Analysis (MCDA).
4. Produce a groundwater potential map.
5. Classify groundwater potential into suitability zones.
6. Identify priority areas for groundwater development.

> **Important interpretation:** The final map represents relative groundwater potential based on the selected criteria and weights. It is a screening/prioritisation product and does not replace hydrogeological investigation, geophysical surveys, borehole logs, pumping tests, or field validation.

### Library Imports


In [1]:
# import libraries
import pandas as pd
import geopandas as gpd
import ee
import geemap

import numpy as np

### Google Earth Engine Configuration


In [ ]:
# Authenticate GEE
ee.Authenticate()

# Initialize GEE
ground_water_project = ""   

ee.Initialize(project=ground_water_project)

## Study Area  (Kano State)


In [4]:
admin1 = ee.FeatureCollection("FAO/GAUL/2015/level1")

kano = admin1.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Nigeria"),
        ee.Filter.eq("ADM1_NAME", "Kano")
    )
)

kano_geometry = kano.geometry()

### Study Area Map

In [5]:
study_map = geemap.Map()
study_map.center_object(kano, 8)
study_map.add_layer(
    kano.style(color="red", fillColor="00000000", width=2),
    {},
    "Kano Boundary"
)
study_map

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…

### Reusable Base Maps


In [6]:
def make_kano_map(zoom=8, add_boundary=True):
    m = geemap.Map()
    m.center_object(kano, zoom)
    if add_boundary:
        m.add_layer(
            kano.style(color="red", fillColor="00000000", width=2),
            {},
            "Kano Boundary"
        )
    return m


overview_map = make_kano_map(8)
detail_map = make_kano_map(10)
result_map = make_kano_map(8)

###  Analysis and Visualization Params


In [7]:
TARGET_CRS = "EPSG:32632"   # WGS 84 / UTM zone 32N
TARGET_SCALE = 30

dem_vis = {
    "min": 300,
    "max": 900,
    "palette": ["0b3d0b", "4f9d4f", "ffff99", "c98b4b", "ffffff"]
}

slope_vis = {
    "min": 0,
    "max": 20,
    "palette": ["ffffff", "ffff00", "ff9900", "ff0000"]
}

landcover_vis = {
    "min": 10,
    "max": 100,
    "palette": [
        "#006400", "#ffbb22", "#ffff4c", "#f096ff",
        "#fa0000", "#b4b4b4", "#f0f0f0", "#0064c8",
        "#0096a0", "#00cf75", "#fae6a0"
    ]
}

soil_vis = {
    "min": 1,
    "max": 12,
    "palette": [
        "#d5c36b", "#b96947", "#9d3706", "#ae868f",
        "#f86714", "#46d143", "#368f20", "#3e5a14",
        "#ffd557", "#fff72e", "#ff5a9d", "#ff005b"
    ]
}

suitability_vis = {
    "min": 1,
    "max": 5,
    "palette": ["red", "orange", "yellow", "lightgreen", "darkgreen"]
}

potential_legend = {
    "Very Low": "red",
    "Low": "orange",
    "Moderate": "yellow",
    "High": "lightgreen",
    "Very High": "darkgreen"
}

##  Thematic Layers

The selected groundwater controlling factors are prepared below. Each layer is clipped to Kano and kept under a descriptive heading.

###  Elevation and Slope

Elevation and slope are derived from the NASA SRTM 30 m DEM. Slope is calculated in degrees. Lower slopes are treated as more favourable for infiltration in the suitability model.

In [8]:
dem = ee.Image("USGS/SRTMGL1_003").select("elevation").clip(kano_geometry)
elevation = dem.rename("elevation")
slope = ee.Terrain.slope(dem).rename("slope")

m = make_kano_map(9)
m.add_layer(elevation, dem_vis, "Elevation")
m.add_layer(slope, slope_vis, "Slope")
m

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…

###  Land Cover

ESA WorldCover v200 provides the 2021 land-cover map at 10 m resolution.

In [9]:
worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
    .clip(kano_geometry)
)

m = make_kano_map(9)
m.add_layer(worldcover, landcover_vis, "ESA WorldCover 2021")
m

Map(center=[11.735699301548118, 8.513584659731869], controls=(WidgetControl(options=['position', 'transparent_…

###  Soil Texture

OpenLandMap USDA soil texture is represented using the topsoil `b0` band.

In [10]:
soil_texture = (
    ee.Image("OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02")
    .select("b0")
    .clip(kano_geometry)
)

m = make_kano_map(9)
m.add_layer(soil_texture, soil_vis, "USDA Soil Texture (0 cm)")
m

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…

### Rainfall  (Mean Annual Precipitation)

CHIRPS daily precipitation is aggregated into annual totals for 2014–2023, and the ten annual totals are averaged to produce mean annual precipitation.

In [11]:
CHIRPS_START_YEAR = 2014
CHIRPS_END_YEAR = 2023

chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")

def annual_precipitation(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")
    return (
        chirps
        .filterDate(start, end)
        .sum()
        .rename("annual_precipitation")
        .set("year", year)
    )

annual_images = [
    annual_precipitation(year)
    for year in range(CHIRPS_START_YEAR, CHIRPS_END_YEAR + 1)
]

annual_precipitation_collection = ee.ImageCollection.fromImages(annual_images)

rainfall = (
    annual_precipitation_collection
    .mean()
    .rename("annual_precipitation")
    .clip(kano_geometry)
)

rainfall_vis = {
    "min": 500,
    "max": 1200,
    "palette": ["f7fbff", "c6dbef", "6baed6", "2171b5", "08306b"]
}

m = make_kano_map(8)
m.add_layer(rainfall, rainfall_vis, "Mean Annual Precipitation (2014–2023)")
m

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…

###  Hydrological Layer  (Drainage Density)

MERIT Hydro provides upstream drainage area at approximately 90 m.

In [12]:
merit_hydro = ee.Image("MERIT/Hydro/v1_0_1")
flow_accumulation = merit_hydro.select("upa").clip(kano_geometry)

STREAM_THRESHOLD_KM2 = 100

stream_mask = flow_accumulation.gte(STREAM_THRESHOLD_KM2).selfMask()

# Approximate drainage density within a 5 km circular neighbourhood.
# MERIT Hydro is approximately 90 m, so each stream pixel is represented
# by an approximate 90 m centreline segment.
window_radius_m = 5000
stream_kernel = ee.Kernel.circle(
    radius=window_radius_m,
    units="meters",
    normalize=False
)

stream_pixel_count = (
    stream_mask
    .unmask(0)
    .reduceNeighborhood(
        reducer=ee.Reducer.sum(),
        kernel=stream_kernel,
        skipMasked=False
    )
)

approx_stream_length_m = stream_pixel_count.multiply(90)
window_area_m2 = 3.141592653589793 * (window_radius_m ** 2)

drainage_density = (
    approx_stream_length_m
    .divide(window_area_m2)
    .multiply(1000)
    .rename("drainage_density")
    .clip(kano_geometry)
)

drainage_density_vis = {
    "min": 0,
    "max": 1,
    "palette": ["ffffff", "9ecae1", "3182bd", "08519c"]
}

m = make_kano_map(9)
m.add_layer(drainage_density, drainage_density_vis, "Drainage Density (km/km²)")
m

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…

###  Topographic Wetness Index (TWI)

TWI is derived as a topographic wetness proxy using MERIT Hydro upstream drainage area and MERIT DEM slope:

`TWI = ln(flow accumulation / tan(slope))`

In [13]:
merit_dem = ee.Image("MERIT/DEM/v1_0_3").select("dem").clip(kano_geometry)
merit_slope = ee.Terrain.slope(merit_dem)

slope_radians = merit_slope.multiply(3.141592653589793 / 180)
tan_slope = slope_radians.tan().max(0.001)

twi = (
    flow_accumulation
    .divide(tan_slope)
    .log()
    .rename("twi")
    .clip(kano_geometry)
)

twi_vis = {
    "min": 0,
    "max": 12,
    "palette": ["white", "yellow", "green", "blue"]
}

m = make_kano_map(9)
m.add_layer(twi, twi_vis, "Topographic Wetness Index")
m

Map(center=[11.735699301548118, 8.513584659731869], controls=(WidgetControl(options=['position', 'transparent_…

##  Reclassification to Groundwater Suitability Scores

All criteria are converted to a common scale from **1 (least suitable)** to **5 (most suitable)** before weighted overlay.


In [14]:
def percentile_reclassify(image, region, scale, band_name, inverse=False):
    percentiles = image.reduceRegion(
        reducer=ee.Reducer.percentile([20, 40, 60, 80]),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e13
    )

    p20 = ee.Number(percentiles.get(f"{band_name}_p20"))
    p40 = ee.Number(percentiles.get(f"{band_name}_p40"))
    p60 = ee.Number(percentiles.get(f"{band_name}_p60"))
    p80 = ee.Number(percentiles.get(f"{band_name}_p80"))

    if not inverse:
        classified = (
            image.lt(p20).multiply(1)
            .add(image.gte(p20).And(image.lt(p40)).multiply(2))
            .add(image.gte(p40).And(image.lt(p60)).multiply(3))
            .add(image.gte(p60).And(image.lt(p80)).multiply(4))
            .add(image.gte(p80).multiply(5))
        )
    else:
        classified = (
            image.gte(p80).multiply(1)
            .add(image.gte(p60).And(image.lt(p80)).multiply(2))
            .add(image.gte(p40).And(image.lt(p60)).multiply(3))
            .add(image.gte(p20).And(image.lt(p40)).multiply(4))
            .add(image.lt(p20).multiply(5))
        )

    return classified.rename(f"{band_name}_suitability")

###  Rainfall Suitability

Higher mean annual precipitation is assigned higher suitability.

In [15]:
rainfall_suitability = percentile_reclassify(
    rainfall,
    kano_geometry,
    5566,
    "annual_precipitation",
    inverse=False
)

### Elevation Suitability

 Lower elevations are assigned higher suitability.

In [16]:
elevation_suitability = (
    elevation.expression(
        "(b('elevation') <= 200) ? 5"
        ": (b('elevation') <= 400) ? 4"
        ": (b('elevation') <= 700) ? 3"
        ": (b('elevation') <= 1000) ? 2"
        ": 1"
    )
    .rename("elevation_suitability")
)

### Slope Suitability

Gentler slopes are assigned higher suitability because they generally favour infiltration relative to steep slopes.

In [17]:
slope_suitability = (
    slope.expression(
        "(b('slope') <= 2) ? 5"
        ": (b('slope') <= 5) ? 4"
        ": (b('slope') <= 10) ? 3"
        ": (b('slope') <= 20) ? 2"
        ": 1"
    )
    .rename("slope_suitability")
)

### Drainage Density Suitability

Lower drainage density is assigned higher groundwater suitability in the MCDA because high drainage density generally indicates greater surface runoff and comparatively less infiltration opportunity.

In [18]:
drainage_suitability = percentile_reclassify(
    drainage_density,
    kano_geometry,
    500,
    "drainage_density",
    inverse=True
)

### TWI Suitability

Higher TWI values indicate greater relative wetness/moisture accumulation and are assigned higher suitability.

In [19]:
twi_suitability = (
    twi.expression(
        "(b('twi') <= 2) ? 1"
        ": (b('twi') <= 4) ? 2"
        ": (b('twi') <= 6) ? 3"
        ": (b('twi') <= 8) ? 4"
        ": 5"
    )
    .rename("twi_suitability")
)

### Land Cover Suitability

Vegetated and cultivated classes receive higher scores.
Built-up receive lower scores.

In [20]:
landcover_suitability = worldcover.remap(
    [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100],
    [5, 4, 4, 5, 1, 2, 1, 1, 4, 5, 2]
).rename("landcover_suitability")

###  Soil Texture Suitability


In [21]:
soil_suitability = soil_texture.remap(
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    [5, 5, 4, 4, 3, 3, 2, 2, 2, 1, 1, 1]
).rename("soil_suitability")

## Suitability Layer Review

The seven standardized criteria are assembled into one dictionary for consistent use in visualization, validation and MCDA.

In [22]:
suitability_layers = {
    "Rainfall": rainfall_suitability,
    "Elevation": elevation_suitability,
    "Slope": slope_suitability,
    "Land Cover": landcover_suitability,
    "Soil Texture": soil_suitability,
    "Drainage Density": drainage_suitability,
    "TWI": twi_suitability
}

In [23]:
suitability_map = make_kano_map(8)

for name, image in suitability_layers.items():
    suitability_map.add_layer(
        image,
        suitability_vis,
        f"{name} Suitability"
    )

suitability_map.add_legend(
    title="Suitability Score",
    legend_dict={
        "1 — Very Low": "red",
        "2 — Low": "orange",
        "3 — Moderate": "yellow",
        "4 — High": "lightgreen",
        "5 — Very High": "darkgreen"
    }
)

suitability_map

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…

## GIS-Based Multi-Criteria Decision Analysis (MCDA)

### Analytic Hierarchy Process (AHP) Weighting

The relative importance of the seven groundwater-potential criteria was determined using the Analytic Hierarchy Process (AHP).

The criteria were compared pairwise using Saaty's fundamental 1–9 comparison scale. A value of 1 indicates equal importance between two criteria, while higher values indicate increasing relative importance of one criterion over another. Reciprocal values are used where the criterion in the row is less important than the criterion in the column.

The pairwise comparison matrix was used to calculate normalized criterion weights. The consistency of the expert judgments was evaluated using the Consistency Index (CI) and Consistency Ratio (CR). A Consistency Ratio below 0.10 was considered acceptable.

The seven criteria considered were rainfall, elevation, slope, land cover, soil texture, drainage density, and Topographic Wetness Index (TWI).

In [24]:
# AHP criteria
criteria = [
    "Rainfall",
    "Elevation",
    "Slope",
    "Land Cover",
    "Soil Texture",
    "Drainage Density",
    "TWI"
]

# Pairwise comparison matrix
# Rows and columns follow the same order as the criteria above.
# Saaty's 1–9 scale is used, with reciprocal values for opposite comparisons.

ahp_matrix = np.array([
    [1,   3,   3,   2,   1,   2,   1],
    [1/3, 1,   2,   1/2, 1/3, 1/2, 1/3],
    [1/3, 1/2, 1,   1/2, 1/3, 1/2, 1/2],
    [1/2, 2,   2,   1,   1/2, 1,   1/2],
    [1,   3,   3,   2,   1,   2,   1],
    [1/2, 2,   2,   1,   1/2, 1,   1/2],
    [1,   3,   2,   2,   1,   2,   1]
], dtype=float)

# Check matrix dimensions
assert ahp_matrix.shape == (7, 7)

# Check reciprocal property
assert np.allclose(
    ahp_matrix * ahp_matrix.T,
    np.ones((7, 7)),
    atol=1e-9
)

# Calculate principal eigenvector
eigenvalues, eigenvectors = np.linalg.eig(ahp_matrix)

max_index = np.argmax(eigenvalues.real)
lambda_max = eigenvalues[max_index].real

weights = np.abs(eigenvectors[:, max_index].real)
weights = weights / weights.sum()

# Consistency Index
n = len(criteria)
CI = (lambda_max - n) / (n - 1)

# Random Index for n = 7
RI = 1.32

# Consistency Ratio
CR = CI / RI

# Display results
# print("AHP Criterion Weights")
# print("-" * 40)

# for criterion, weight in zip(criteria, weights):
#     print(f"{criterion:<20} {weight:.4f} ({weight * 100:.2f}%)")

# print("\nSum of weights:", weights.sum())
# print("Principal eigenvalue (λmax):", round(lambda_max, 4))
# print("Consistency Index (CI):", round(CI, 4))
# print("Consistency Ratio (CR):", round(CR, 4))

# if CR < 0.10:
#    print("\nAHP consistency check: ACCEPTABLE (CR < 0.10)")
# else:
#     print("\nAHP consistency check: NOT ACCEPTABLE (CR >= 0.10)")

"Criterion weights were derived using AHP through pairwise comparison of the seven selected groundwater-potential factors. The resulting comparison matrix produced a consistency ratio of 0.0139, which is below the accepted threshold of 0.10."

### AHP-Derived Criterion Weights

The AHP procedure produced normalized weights for the seven groundwater-potential criteria. Rainfall and soil texture received the highest weights at 21.25% each, followed by TWI at 20.34%. Land cover and drainage density each received 11.61%, while elevation and slope received 7.45% and 6.49%, respectively.

The weights sum to 1.00 and the calculated Consistency Ratio is 0.0139, indicating an acceptable level of consistency in the pairwise comparisons.

In [25]:
# Convert the AHP weights into a dictionary
ahp_weights = dict(zip(criteria, weights))

# Confirm that every suitability layer has an AHP weight
assert set(suitability_layers.keys()) == set(ahp_weights.keys())

# # Display the final weights used in the MCDA
# print("Final AHP Weights Used in MCDA")
# print("-" * 40)

# for criterion, weight in ahp_weights.items():
#     print(f"{criterion:<20} {weight:.4f} ({weight * 100:.2f}%)")

# print("\nWeight total:", sum(ahp_weights.values()))
# print("Consistency Ratio:", round(CR, 4))

## Weighted Groundwater Potential Index

The standardized suitability layers were combined using a weighted linear combination based on the AHP-derived criterion weights.

For each location, the groundwater potential index is calculated by multiplying the suitability score of each criterion by its corresponding AHP weight and summing the resulting weighted scores.

The resulting index represents the relative groundwater potential across Kano State. Because the suitability scores range from 1 to 5 and the AHP weights sum to 1.00, the theoretical range of the weighted index is also 1 to 5.

In [26]:
# Weighted Linear Combination
# Calculate the Groundwater Potential Index (GWPI)

groundwater_index = ee.Image(0)

for criterion, weight in ahp_weights.items():
    groundwater_index = groundwater_index.add(
        suitability_layers[criterion].multiply(weight)
    )

groundwater_index = (
    groundwater_index
    .clip(kano)
    .rename("GWPI")
)

# print("Groundwater Potential Index created successfully.")

In [29]:
# Calculate the GWPI range using a coarser scale for diagnostics.
# This is only for inspecting the index range, not for producing the final map.

gwpi_stats = groundwater_index.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=kano.geometry(),
    scale=1000,
    bestEffort=True,
    maxPixels=1e8
)

print("GWPI range:")
print(gwpi_stats.getInfo())

GWPI range:
{'GWPI_max': 4.17416757904275, 'GWPI_min': 1.7373345142481436}


## Groundwater Potential Classification

The Groundwater Potential Index (GWPI) was classified into five groundwater potential zones: Very Low, Low, Moderate, High, and Very High.

The observed GWPI values across Kano State ranged from approximately 1.74 to 4.17. Equal-interval classification was applied across this observed range to divide the index into five potential classes.

The resulting classes represent relative groundwater potential within the study area and should be interpreted as groundwater-potential zones rather than direct measurements of aquifer productivity.

In [ ]:
# Observed GWPI range
GWPI_MIN = 1.7373345142481436
GWPI_MAX = 4.17416757904275

# Calculate equal interval
GWPI_INTERVAL = (GWPI_MAX - GWPI_MIN) / 5

# Classification thresholds
class_2 = GWPI_MIN + GWPI_INTERVAL
class_3 = GWPI_MIN + (2 * GWPI_INTERVAL)
class_4 = GWPI_MIN + (3 * GWPI_INTERVAL)
class_5 = GWPI_MIN + (4 * GWPI_INTERVAL)

# print("GWPI Classification Thresholds")
# print("-" * 40)
# print(f"Very Low:   {GWPI_MIN:.4f} – {class_2:.4f}")
# print(f"Low:        {class_2:.4f} – {class_3:.4f}")
# print(f"Moderate:   {class_3:.4f} – {class_4:.4f}")
# print(f"High:       {class_4:.4f} – {class_5:.4f}")
# print(f"Very High:  {class_5:.4f} – {GWPI_MAX:.4f}")

GWPI Classification Thresholds
----------------------------------------
Very Low:   1.7373 – 2.2247
Low:        2.2247 – 2.7121
Moderate:   2.7121 – 3.1994
High:       3.1994 – 3.6868
Very High:  3.6868 – 4.1742


In [ ]:
# Classify the Groundwater Potential Index into five zones

groundwater_potential = (
    ee.Image(1)
    .where(groundwater_index.gte(class_2), 2)
    .where(groundwater_index.gte(class_3), 3)
    .where(groundwater_index.gte(class_4), 4)
    .where(groundwater_index.gte(class_5), 5)
    .rename("Groundwater_Potential")
    .clip(kano)
)

# print("Groundwater potential classification created successfully.")

Groundwater potential classification created successfully.


In [40]:
potential_vis = {
    "min": 1,
    "max": 5,
    "palette": [
        "red",
        "orange",
        "yellow",
        "lightgreen",
        "darkgreen"
    ]
}

potential_map = make_kano_map(8)

potential_map.add_layer(
    groundwater_potential,
    potential_vis,
    "Groundwater Potential Zones"
)

potential_map.add_legend(
    title="Groundwater Potential",
    legend_dict={
        "1 — Very Low": "red",
        "2 — Low": "orange",
        "3 — Moderate": "yellow",
        "4 — High": "lightgreen",
        "5 — Very High": "darkgreen"
    }
)

potential_map

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…